## Detectando casos de COVID a partir de imagens de Tomografia


Esse é um projeto de pós graduação em inteligência artificial cujo o objetivo deste conjunto de dados é incentivar a pesquisa e o desenvolvimento de métodos de
inteligência artificial capazes de identificar se uma pessoa está infectada pelo SARS-CoV-2 por
meio da análise de suas tomografias computadorizadas.

O dataset em questão está disponível no kaggle e foi coletado de pacientes reais em hospitais
no estado de São Paulo.

Nesse projeto, as principais habilidades a serem exercitadas estão relacionadas ao
Processamento de Imagens e a utilização de modelos de Deep Learning para classificação

fonte de dados: https://www.kaggle.com/datasets/plameneduardo/sarscov2-ctscan-dataset

### Carregamento e Pré-processamento básico para normalizar imagens

In [16]:
# importando as blibliotecas
import os
from PIL import Image
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

In [17]:
# bibliotecas que eu não conhecia até o momento

# PIL (Python Imaging Library) é uma biblioteca para abrir, manipular e salvar muitos formatos de imagem diferentes.
# cv2 é a biblioteca OpenCV, que é usada para processamento de imagens e visão computacional.

In [18]:
# Definindo o caminho para a pasta de imagens
covid = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\COVID'
no_covid = r'C:\Users\geova\OneDrive\Ambiente de Trabalho\inteligencia artificial\5 machine learning\detectando casos de covid\non-COVID'


In [19]:
# definindo um tamanho padrão pois quando tentei converter lá embaixo as listas de imagens para arrays, algumas imagens tinham tamanhos diferentes e isso dá erro no codigo
img_altura, img_largura = 150, 150

In [20]:
# lista para armazenar as imagens e labels das imagens que SÃO de COVID 
imagens = []
labels = []

In [21]:
# lista para armazenar as imagens e labels das imagens que SÃO de COVID
for filename in os.listdir(covid):
    if filename.endswith('.png'): # verifica se no final do arquivo tem .png
        img_path = os.path.join(covid, filename) # cria o caminho completo do arquivo
        try:
            # abre a imagem e converte para RGB porque algumas imagens tinham mais 3 de canais provavelmente RGBA e quanto tem o quarto canal que é o alpha, o PIL não consegue converter para array
            img = Image.open(img_path).convert('RGB')  # converte a imagem para RGB ou seja 3 canais
            img_redimensionada = img.resize((img_largura, img_altura))  # redimensiona a imagem para o tamanho padrão
            img_array = np.array(img_redimensionada)  # converte a imagem para um array numpy   
            # print(f"shape da imagem: {filename} (covid): {img_array.shape}") # aqui eu tinha um dado um print só pra vê quantos canais tinha a imagem
            imagens.append(img_array)  # adiciona a imagem à lista de imagens
            labels.append(1)  # adiciona o label 1 para indicar que é COVID
        except Exception as e:
            print(f'Erro ao processar a imagem {filename}: {e}')

In [22]:
# lista para armazenar as imagens e labels das imagens que NÃO SÃO de COVID
for filename in os.listdir(no_covid):
    if filename.endswith('.png'):  # verifica se no final do arquivo tem .png
        img_path = os.path.join(no_covid, filename)  # cria o caminho completo do arquivo
        try:
            img = Image.open(img_path).convert('RGB')  # abre a imagem e converte para RGB
            img_redimensionada = img.resize((img_largura, img_altura))  # redimensiona a imagem para o tamanho padrão  
            img_array = np.array(img_redimensionada)  # converte a imagem para um array numpy
            # print(f"shape da imagem: {filename} (no covid): {img_array.shape}") # aqui também tinha dado um print só pra vê quantos canais tinha a imagem
            imagens.append(img_array)  # adiciona a imagem à lista de imagens
            labels.append(0)  # adiciona a label 0 para indicar que não é COVID
        except Exception as e:
            print(f"Erro ao processar a imagem {filename}: {e}")


In [23]:
imagens = np.array(imagens)  # converte a lista de imagens para um array numpy
labels = np.array(labels)  # converte a lista de labels para um array numpy 

### Normalizando os valores dos pixels

In [24]:
imagens = np.array(imagens, dtype=np.float32) # converte o array de imagens para o tipo float32 para evitar problemas de precisão

imagens_normalizadas = imagens / 255.0

print(f"Formato do array de imagens normalizadas: {imagens_normalizadas.shape}")
print(f"Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): {imagens_normalizadas[0][0][0]}")

Formato do array de imagens normalizadas: (2481, 150, 150, 3)
Valores dos pixels após a normalização (exemplo - primeiro pixel da primeira imagem): [0.7529412 0.7529412 0.7529412]


### Aplicando o Cross Validation

In [25]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.linear_model import LogisticRegression 

In [26]:
# Create a simple model (you'll replace this with your actual model later)
model = LogisticRegression(solver='liblinear')

# Define the number of folds for cross-validation (e.g., 5 or 10)
k_folds = 5

# Create a KFold object
kf = KFold(n_splits=k_folds, shuffle=True, random_state=42)

# Perform cross-validation
# cross_val_score returns an array of scores obtained on each fold
cv_scores = cross_val_score(model, imagens.reshape(imagens.shape[0], -1), labels, cv=kf, scoring='accuracy')

print(f"Scores de Validação Cruzada (Acurácia) para cada fold: {cv_scores}")
print(f"Acurácia Média de Validação Cruzada: {np.mean(cv_scores)}")

Scores de Validação Cruzada (Acurácia) para cada fold: [0.77464789 0.7983871  0.81653226 0.79233871 0.79233871]
Acurácia Média de Validação Cruzada: 0.7948489323034984


In [27]:
import tensorflow as tf
from tensorflow.keras import layers, models

# Definindo o formato de entrada das imagens (altura, largura, canais)
input_shape = (150, 150, 3) # Usamos 3 canais porque convertemos as imagens para RGB

# Criando o modelo
model = models.Sequential([
    layers.Conv2D(32, (3, 3), activation='relu', input_shape=input_shape),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dense(1, activation='sigmoid') # Camada de saída com 1 neurônio e função de ativação sigmoid para classificação binária (0 ou 1)
])

# Exibindo um resumo do modelo para ver sua arquitetura
model.summary()

ModuleNotFoundError: No module named 'tensorflow.python'